# Experiment 4: k-Nearest Neighbors and the curse of dimensionality

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

wine = load_wine()
X_tr, X_te, y_tr, y_te = train_test_split(wine.data, wine.target, test_size=0.3,
                                          random_state=0, stratify=wine.target)

In [ ]:
# Tune k with 5-fold cross-validation (scaling is done inside the pipeline)
ks = list(range(1, 31))
cv_scores = []
for k in ks:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    cv_scores.append(cross_val_score(pipe, X_tr, y_tr, cv=5).mean())

best_k = ks[int(np.argmax(cv_scores))]
print("Best k:", best_k, " CV accuracy:", round(max(cv_scores), 3))

plt.plot(ks, cv_scores, marker="o")
plt.axvline(best_k, color="red", linestyle="--")
plt.xlabel("k")
plt.ylabel("CV accuracy")
plt.title("Choosing k for kNN")
plt.show()

final = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)).fit(X_tr, y_tr)
print("Test accuracy with best k:", round(final.score(X_te, y_te), 3))

In [ ]:
# Adding irrelevant noisy features hurts kNN
rng = np.random.default_rng(0)
for n_noise in [0, 10, 50, 100]:
    Xn_tr = np.hstack([X_tr, rng.normal(size=(len(X_tr), n_noise))])
    Xn_te = np.hstack([X_te, rng.normal(size=(len(X_te), n_noise))])
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)).fit(Xn_tr, y_tr)
    print(f"Noise features={n_noise:3d}  test acc={m.score(Xn_te, y_te):.3f}")

In [ ]:
# Curse of dimensionality: distances between random points become almost equal
dims = [2, 5, 10, 50, 100, 500, 1000]
ratios = []
for d in dims:
    pts = rng.uniform(size=(500, d))
    q = rng.uniform(size=d)
    dist = np.linalg.norm(pts - q, axis=1)
    ratios.append((dist.max() - dist.min()) / dist.min())

plt.plot(dims, ratios, marker="o")
plt.xscale("log")
plt.xlabel("dimensions")
plt.ylabel("(max - min) / min distance")
plt.title("Curse of dimensionality")
plt.show()